# Multi-format model extension benchmark

Train one `LogisticRegression` on Iris (tree models fail ONNX export with current sklearn/skl2onnx), export to each extension folder (`pkl`, `joblib`, `dill`, `onnx`, `pmml`), run each folder's `inference.py`, and report accuracy.

In [ ]:
import json
import pickle
import subprocess
from pathlib import Path

import dill
import joblib
import numpy as np
import pandas as pd
from nyoka.skl.skl_to_pmml import skl_to_pmml
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ROOT = Path('.').resolve().parent  # notebook lives in pkl/
EXTENSIONS = ['pkl', 'joblib', 'dill', 'onnx', 'pmml']
RANDOM_STATE = 42

with open(ROOT / 'pkl' / 'schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)
FEATURE_COLS = schema['input_parameters_name']
print('Feature columns:', FEATURE_COLS)

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=FEATURE_COLS)
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

model = LogisticRegression(max_iter=500, random_state=RANDOM_STATE)
model.fit(X_train, y_train)

train_acc = accuracy_score(y_train, model.predict(X_train))
holdout_acc = accuracy_score(y_test, model.predict(X_test))
print(f'Train accuracy: {train_acc:.4f}')
print(f'Holdout accuracy (sklearn): {holdout_acc:.4f}')

In [ ]:
INPUT_WITH_NOISE = ROOT / 'pkl' / 'dataset' / 'input_with_random_column.csv'


def export_models(trained_model) -> None:
    (ROOT / 'pkl' / 'model').mkdir(parents=True, exist_ok=True)
    with open(ROOT / 'pkl' / 'model' / 'model.pkl', 'wb') as f:
        pickle.dump(trained_model, f)

    (ROOT / 'joblib' / 'model').mkdir(parents=True, exist_ok=True)
    joblib.dump(trained_model, ROOT / 'joblib' / 'model' / 'model.joblib')

    (ROOT / 'dill' / 'model').mkdir(parents=True, exist_ok=True)
    with open(ROOT / 'dill' / 'model' / 'model.dill', 'wb') as f:
        dill.dump(trained_model, f)

    (ROOT / 'onnx' / 'model').mkdir(parents=True, exist_ok=True)
    initial_type = [('float_input', FloatTensorType([None, len(FEATURE_COLS)]))]
    onnx_model = convert_sklearn(trained_model, initial_types=initial_type)
    with open(ROOT / 'onnx' / 'model' / 'model.onnx', 'wb') as f:
        f.write(onnx_model.SerializeToString())

    (ROOT / 'pmml' / 'model').mkdir(parents=True, exist_ok=True)
    pmml_pipeline = Pipeline([('classifier', trained_model)])
    skl_to_pmml(
        pmml_pipeline,
        FEATURE_COLS,
        target_name='target',
        pmml_f_name=str(ROOT / 'pmml' / 'model' / 'model.pmml'),
    )


def deploy_input_csv() -> None:
    for ext in EXTENSIONS:
        out_dir = ROOT / ext / 'dataset'
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / 'input.csv').write_bytes(INPUT_WITH_NOISE.read_bytes())


train_idx = X_train.index.to_numpy()
test_idx = X_test.index.to_numpy()
split_info = {
    'train_indices': train_idx.tolist(),
    'test_indices': test_idx.tolist(),
    'train_accuracy': float(train_acc),
    'test_accuracy': float(holdout_acc),
}
with open(ROOT / 'pkl' / 'dataset' / 'train_test_split.json', 'w', encoding='utf-8') as f:
    json.dump(split_info, f, indent=2)

export_models(model)
deploy_input_csv()
print('Deployed', INPUT_WITH_NOISE, 'as input.csv for:', EXTENSIONS)

In [ ]:
with open(ROOT / 'pkl' / 'dataset' / 'train_test_split.json', 'r', encoding='utf-8') as f:
    split = json.load(f)
train_idx = np.array(split['train_indices'])
test_idx = np.array(split['test_indices'])

results = []
for ext in EXTENSIONS:
    folder = ROOT / ext
    output_path = folder / 'output' / 'output.csv'
    output_path.parent.mkdir(parents=True, exist_ok=True)

    proc = subprocess.run(
        ['python', 'inference.py'],
        cwd=folder,
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f'Inference failed for {ext}\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}'
        )

    preds = pd.read_csv(output_path)['target'].to_numpy()
    results.append({
        'extension': ext,
        'train_accuracy': round(train_acc, 4),
        'test_accuracy': round(holdout_acc, 4),
        'inference_train_accuracy': round(accuracy_score(y[train_idx], preds[train_idx]), 4),
        'inference_test_accuracy': round(accuracy_score(y[test_idx], preds[test_idx]), 4),
        'inference_full_accuracy': round(accuracy_score(y, preds), 4),
        'rows': len(preds),
    })

results_df = pd.DataFrame(results)
results_df.to_csv(ROOT / 'pkl' / 'output' / 'accuracy_report.csv', index=False)
results_df

In [ ]:
results_df['matches_pkl_test'] = results_df['inference_test_accuracy'] == results_df.loc[results_df['extension'] == 'pkl', 'inference_test_accuracy'].iloc[0]
results_df